# EPOWER Energy Intelligence Demo Setup

**A Hands-On Lab for Snowflake Intelligence**

---

## What is Snowflake Intelligence?

**Snowflake Intelligence** is a ready-to-use agentic AI application with an intuitive, conversational interface that helps business users discover and act on deep insights. It lets users interact with their structured and unstructured enterprise data using natural language.

Snowflake Intelligence uses AI-powered **data agents** to:
- **Understand** questions in natural language
- **Perform** analysis across structured and unstructured data
- **Generate** trusted insights with full traceability
- **Take action** on behalf of users

It bridges the gap between valuable enterprise data and the people who need it, empowering users to move beyond stale dashboards and rigid reports - finding answers independently while respecting Snowflake's robust security and governance policies.

### How It Works

```
User Question ──► Cortex Agent API ──► Orchestration (LLM) ──► Tool Selection
                                                                     │
                      ┌──────────────────┬──────────────────┬────────┘
                      ▼                  ▼                  ▼
              ┌──────────────┐   ┌──────────────┐   ┌──────────────┐
              │CORTEX ANALYST│   │CORTEX SEARCH │   │ CUSTOM TOOLS │
              │ (Text-to-SQL)│   │    (RAG)     │   │  (Functions) │
              │              │   │              │   │              │
              │ Semantic     │   │ Documents &  │   │ Web Scraper  │
              │ Views        │   │ Unstructured │   │ Procedures   │
              └──────────────┘   └──────────────┘   └──────────────┘
                      │                  │                  │
                      └──────────────────┴──────────────────┘
                                         │
                                         ▼
                            Reflection & Response Generation
```

---

## What You'll Build

This notebook guides you through building a complete **Snowflake Intelligence** demo for **EPOWER**, a fictional German Energy Retail (B2C) use case. You will create:

| Component | Purpose |
|-----------|---------|
| **Governed Data Foundation** | Structured tables managed and secured by Snowflake |
| **Semantic Views** | Business context layer enabling natural language queries |
| **Cortex Search Services** | RAG over documents and service tickets |
| **Intelligence Agent** | Autonomous AI agent orchestrating all capabilities |

### How to Use This Notebook

You can either:
- **Run all cells** sequentially for a quick setup (~10 minutes)
- **Run cell-by-cell** to learn about each concept as you go

### Prerequisites

- ACCOUNTADMIN role access (or equivalent privileges)
- A Snowflake account with Cortex features enabled

## Session Setup

Before running any cells, we need to establish a connection to Snowflake. In a Snowflake Workspace, use `get_active_session()` to get the current session context.

In [3]:
# Import python packages
import pandas as pd

# Snowpark
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

# Cortex Functions
import snowflake.cortex as cortex

# Get the active session
session = get_active_session()

## Table of Contents

1. Introduction to EPOWER Demo
2. Role & Warehouse Setup
3. Database & Schema Creation
4. Internal Stage & Data Upload
5. Data Model - Dimension Tables
6. Data Model - Fact Tables
7. Data Loading from Internal Stage
8. Semantic Views for Cortex Analyst
9. Unstructured Data Processing
10. Cortex Search Services
11. Snowflake Intelligence Agent
12. Verification & Next Steps

In [ ]:
%%sql -r warehouse_result
-- Create a dedicated role for the demo (must be done by ACCOUNTADMIN)
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS Energy_Intelligence_Demo;

-- Grant role to SYSADMIN for administration
GRANT ROLE Energy_Intelligence_Demo TO ROLE SYSADMIN;

-- Grant necessary privileges to the role
GRANT CREATE DATABASE ON ACCOUNT TO ROLE Energy_Intelligence_Demo;
GRANT CREATE WAREHOUSE ON ACCOUNT TO ROLE Energy_Intelligence_Demo;

In [ ]:
current_user = session.get_current_user()

session.sql(f"GRANT ROLE Energy_Intelligence_Demo TO USER {current_user}").collect()
session.sql("USE ROLE Energy_Intelligence_Demo").collect()

print(f"✓ Granted role to user {current_user} and switched to Energy_Intelligence_Demo")

---

## 3. Database & Schema Creation

### Concept: Database Organization

Snowflake uses a three-tier namespace: `DATABASE.SCHEMA.OBJECT`

```
ENERGY_AI_DEMO (Database)
    └── ENERGY_SCHEMA (Schema)
            ├── Tables
            ├── Semantic Views
            ├── Cortex Search Services
            └── Functions/Procedures
```

In [ ]:
%%sql -r db_setup_result
-- Create database and schema
CREATE OR REPLACE DATABASE ENERGY_AI_DEMO;
USE DATABASE ENERGY_AI_DEMO;

CREATE SCHEMA IF NOT EXISTS ENERGY_SCHEMA;
USE SCHEMA ENERGY_SCHEMA;

### File Format Definition

A **File Format** defines how Snowflake should parse incoming data files. This is essential for CSV ingestion.

In [ ]:
%%sql -r file_format_result
-- Create file format for CSV files
CREATE OR REPLACE FILE FORMAT CSV_FORMAT
    TYPE = 'CSV'
    FIELD_DELIMITER = ','
    RECORD_DELIMITER = '\n'
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    TRIM_SPACE = TRUE
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE
    ESCAPE = 'NONE'
    ESCAPE_UNENCLOSED_FIELD = '\\'
    DATE_FORMAT = 'YYYY-MM-DD'
    TIMESTAMP_FORMAT = 'YYYY-MM-DD HH24:MI:SS'
    NULL_IF = ('NULL', 'null', '', 'N/A', 'n/a');

---

## 4. Internal Stage & Data Upload

### Why Do We Need an Internal Stage?

In a Snowflake Workspace, files from your repository are accessible to **Python code** via local paths (e.g., `pd.read_csv("data/file.csv")`). However, **SQL commands** like `COPY INTO` and Cortex functions like `PARSE_DOCUMENT` can only read files from **Snowflake stages** - they cannot access workspace local files directly.

Therefore, we create a single internal stage with two subfolders:

```
@ENERGY_STAGE/
    ├── demo_data/           <- Structured data (CSV files)
    │   ├── customer_dim.csv
    │   ├── product_dim.csv
    │   ├── sales_fact.csv
    │   └── ...
    └── unstructured_docs/   <- Unstructured data (PDFs, Markdown)
        ├── energy/
        ├── products/
        └── service/
```

**This approach enables:**
- `COPY INTO` to load CSV data into tables
- `PARSE_DOCUMENT` to extract text from PDFs
- `CORTEX SEARCH` to index document content
- Consistent file access for all SQL-based operations

In [ ]:
%%sql -r stage_result
-- Create internal stage with directory tables enabled
CREATE OR REPLACE STAGE ENERGY_STAGE
    FILE_FORMAT = CSV_FORMAT
    COMMENT = 'Internal stage for structured (CSV) and unstructured (PDF/MD) data'
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

In [ ]:
import os
import glob

# Get the workspace root directory (parent of notebooks folder)
workspace_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
print(f"Workspace root: {workspace_root}\n")

# Upload structured data (CSV files)
print("Uploading structured data (CSV files)...")
csv_path = os.path.join(workspace_root, "demo_data", "*.csv")
csv_files = glob.glob(csv_path)
print(f"Found {len(csv_files)} CSV files")
for file_path in csv_files:
    file_name = os.path.basename(file_path)
    session.file.put(file_path, "@ENERGY_STAGE/demo_data/", auto_compress=False, overwrite=True)
    print(f"  ✓ Uploaded: {file_name}")

# Upload unstructured documents (PDF/MD files)
print("\nUploading unstructured documents (PDF/MD files)...")
docs_path = os.path.join(workspace_root, "unstructured_docs", "**", "*")
doc_files = [f for f in glob.glob(docs_path, recursive=True) if f.endswith(('.pdf', '.md'))]
print(f"Found {len(doc_files)} document files")
for file_path in doc_files:
    file_name = os.path.basename(file_path)
    session.file.put(file_path, "@ENERGY_STAGE/unstructured_docs/", auto_compress=False, overwrite=True)
    print(f"  ✓ Uploaded: {file_name}")

session.sql("ALTER STAGE ENERGY_STAGE REFRESH").collect()
print(f"\n✓ Stage refreshed - {len(csv_files)} CSV files and {len(doc_files)} documents uploaded")

---

## 5. Data Model - Dimension Tables

### Concept: Star Schema Design

We use a **Star Schema** - a classic data warehouse design pattern:

- **Dimension Tables**: Descriptive attributes (WHO, WHAT, WHERE, WHEN)
- **Fact Tables**: Measurable events/transactions (HOW MUCH, HOW MANY)

**Benefits:**
- Simple, intuitive queries
- Optimized for analytical workloads
- Perfect for Semantic Views and Cortex Analyst

Our EPOWER demo has **13 dimension tables**:

| Dimension Table | Business Domain |
|-----------------|----------------|
| `product_category_dim` | Energy product categories |
| `product_dim` | Products and tariffs |
| `customer_dim` | Residential and business customers |
| `vendor_dim` | Installation and service partners |
| `account_dim` | Financial accounts |
| `department_dim` | Organizational departments |
| `region_dim` | German geographic regions |
| `sales_rep_dim` | Energy consultants |
| `campaign_dim` | Marketing campaigns |
| `channel_dim` | Sales channels |
| `employee_dim` | Employees |
| `job_dim` | Job positions |
| `location_dim` | Office locations |

In [ ]:
%%sql -r dim_tables_result
-- ==========================================
-- DIMENSION TABLES
-- ==========================================

-- Product Category Dimension
CREATE OR REPLACE TABLE product_category_dim (
    category_key INT PRIMARY KEY,
    category_name VARCHAR(100) NOT NULL,
    vertical VARCHAR(50) NOT NULL
) COMMENT = 'Energy product categories: Electricity, Gas, Solar, Heat Pumps, Smart Home, E-Mobility';

-- Product Dimension (Energy products/tariffs)
CREATE OR REPLACE TABLE product_dim (
    product_key INT PRIMARY KEY,
    product_name VARCHAR(200) NOT NULL,
    category_key INT NOT NULL,
    category_name VARCHAR(100),
    vertical VARCHAR(50)
) COMMENT = 'Energy products: EPOWER Strom, Gas, Solar, Wärmepumpe offerings';

-- Customer Dimension (German residential and business customers)
CREATE OR REPLACE TABLE customer_dim (
    customer_key INT PRIMARY KEY,
    customer_name VARCHAR(200) NOT NULL,
    customer_type VARCHAR(50),
    housing_type VARCHAR(100),
    vertical VARCHAR(50),
    address VARCHAR(200),
    city VARCHAR(100),
    state VARCHAR(50),
    zip VARCHAR(20),
    region_key INT
) COMMENT = 'German energy customers with housing type for consumption analysis';

-- Vendor Dimension (Installation and service partners)
CREATE OR REPLACE TABLE vendor_dim (
    vendor_key INT PRIMARY KEY,
    vendor_name VARCHAR(200) NOT NULL,
    vendor_type VARCHAR(100),
    vertical VARCHAR(50),
    address VARCHAR(200),
    city VARCHAR(100),
    state VARCHAR(50),
    zip VARCHAR(20)
) COMMENT = 'Installation partners, service providers, and suppliers';

-- Account Dimension
CREATE OR REPLACE TABLE account_dim (
    account_key INT PRIMARY KEY,
    account_name VARCHAR(100) NOT NULL,
    account_type VARCHAR(50)
);

-- Department Dimension
CREATE OR REPLACE TABLE department_dim (
    department_key INT PRIMARY KEY,
    department_name VARCHAR(100) NOT NULL
) COMMENT = 'EPOWER organizational departments';

-- Region Dimension
CREATE OR REPLACE TABLE region_dim (
    region_key INT PRIMARY KEY,
    region_name VARCHAR(100) NOT NULL
) COMMENT = 'German geographic regions: North, South, West, East';

-- Sales Rep Dimension
CREATE OR REPLACE TABLE sales_rep_dim (
    sales_rep_key INT PRIMARY KEY,
    rep_name VARCHAR(200) NOT NULL,
    hire_date DATE
) COMMENT = 'Energy consultants and sales representatives';

-- Campaign Dimension
CREATE OR REPLACE TABLE campaign_dim (
    campaign_key INT PRIMARY KEY,
    campaign_name VARCHAR(300) NOT NULL,
    objective VARCHAR(100)
) COMMENT = 'Marketing campaigns for energy products';

-- Channel Dimension
CREATE OR REPLACE TABLE channel_dim (
    channel_key INT PRIMARY KEY,
    channel_name VARCHAR(100) NOT NULL
);

-- Employee Dimension
CREATE OR REPLACE TABLE employee_dim (
    employee_key INT PRIMARY KEY,
    employee_name VARCHAR(200) NOT NULL,
    gender VARCHAR(1),
    hire_date DATE
);

-- Job Dimension
CREATE OR REPLACE TABLE job_dim (
    job_key INT PRIMARY KEY,
    job_title VARCHAR(100) NOT NULL,
    job_level INT
);

-- Location Dimension
CREATE OR REPLACE TABLE location_dim (
    location_key INT PRIMARY KEY,
    location_name VARCHAR(200) NOT NULL
);

---

## 6. Data Model - Fact Tables

### Concept: Fact Tables

Fact tables contain:
- **Metrics/Measures**: Numeric values (amount, quantity, consumption)
- **Foreign Keys**: Links to dimension tables
- **Timestamps**: When the event occurred

Our EPOWER demo has **6 fact tables** covering:

| Fact Table | Business Domain |
|------------|----------------|
| `sales_fact` | Energy contracts and sales |
| `billing_history` | Consumption and invoicing |
| `service_logs` | Customer service tickets |
| `finance_transactions` | Financial operations |
| `marketing_campaign_fact` | Campaign performance |
| `hr_employee_fact` | Human resources |

In [ ]:
%%sql -r fact_tables_result
-- ==========================================
-- FACT TABLES
-- ==========================================

-- Sales Fact Table (Energy contracts)
CREATE OR REPLACE TABLE sales_fact (
    sale_id INT PRIMARY KEY,
    date DATE NOT NULL,
    customer_key INT NOT NULL,
    product_key INT NOT NULL,
    sales_rep_key INT NOT NULL,
    region_key INT NOT NULL,
    vendor_key INT NOT NULL,
    amount DECIMAL(12,2) NOT NULL,
    units INT NOT NULL
) COMMENT = 'Energy contracts and product sales. Amount in EUR, Units = kWh for tariffs or count for hardware';

-- Billing History Table (Energy-specific)
CREATE OR REPLACE TABLE billing_history (
    billing_id INT PRIMARY KEY,
    customer_key INT NOT NULL,
    billing_date DATE NOT NULL,
    billing_type VARCHAR(50) NOT NULL,
    consumption_kwh INT NOT NULL,
    amount DECIMAL(10,2) NOT NULL,
    payment_status VARCHAR(50)
) COMMENT = 'Monthly energy consumption and billing. billing_type = Electricity or Gas';

-- Service Logs Table (Customer service tickets)
CREATE OR REPLACE TABLE service_logs (
    log_id INT PRIMARY KEY,
    customer_key INT NOT NULL,
    log_date DATE NOT NULL,
    topic VARCHAR(100) NOT NULL,
    category VARCHAR(100),
    description VARCHAR(500),
    sentiment VARCHAR(50),
    channel VARCHAR(50),
    priority VARCHAR(50),
    resolution_date DATE,
    agent_key INT
) COMMENT = 'Customer service tickets. Sentiment = Positiv/Neutral/Negativ';

-- Finance Transactions Table
CREATE OR REPLACE TABLE finance_transactions (
    transaction_id INT PRIMARY KEY,
    date DATE NOT NULL,
    account_key INT NOT NULL,
    department_key INT NOT NULL,
    vendor_key INT NOT NULL,
    product_key INT NOT NULL,
    customer_key INT NOT NULL,
    amount DECIMAL(12,2) NOT NULL,
    approval_status VARCHAR(20) DEFAULT 'Pending',
    procurement_method VARCHAR(50),
    approver_id INT,
    approval_date DATE,
    purchase_order_number VARCHAR(50),
    contract_reference VARCHAR(100)
);

-- Marketing Campaign Fact Table
CREATE OR REPLACE TABLE marketing_campaign_fact (
    campaign_fact_id INT PRIMARY KEY,
    date DATE NOT NULL,
    campaign_key INT NOT NULL,
    product_key INT NOT NULL,
    channel_key INT NOT NULL,
    region_key INT NOT NULL,
    spend DECIMAL(10,2) NOT NULL,
    leads_generated INT NOT NULL,
    impressions INT NOT NULL
);

-- HR Employee Fact Table
CREATE OR REPLACE TABLE hr_employee_fact (
    hr_fact_id INT PRIMARY KEY,
    date DATE NOT NULL,
    employee_key INT NOT NULL,
    department_key INT NOT NULL,
    job_key INT NOT NULL,
    location_key INT NOT NULL,
    salary DECIMAL(10,2) NOT NULL,
    attrition_flag INT NOT NULL
);

-- Customer Products Table (Product Ownership - enables cross-domain analysis)
CREATE OR REPLACE TABLE customer_products (
    customer_product_id INT PRIMARY KEY,
    customer_key INT NOT NULL,
    product_key INT NOT NULL,
    category_key INT NOT NULL,
    category_name VARCHAR(100) NOT NULL,
    acquisition_date DATE NOT NULL,
    status VARCHAR(20) DEFAULT 'Active'
) COMMENT = 'Tracks which products each customer owns. Enables queries like avg consumption for heat pump customers';

-- ==========================================
-- SALESFORCE CRM TABLES
-- ==========================================

CREATE OR REPLACE TABLE sf_accounts (
    account_id VARCHAR(20) PRIMARY KEY,
    account_name VARCHAR(200) NOT NULL,
    customer_key INT NOT NULL,
    industry VARCHAR(100),
    vertical VARCHAR(50),
    billing_street VARCHAR(200),
    billing_city VARCHAR(100),
    billing_state VARCHAR(50),
    billing_postal_code VARCHAR(20),
    account_type VARCHAR(50),
    annual_revenue DECIMAL(15,2),
    employees INT,
    created_date DATE
);

CREATE OR REPLACE TABLE sf_opportunities (
    opportunity_id VARCHAR(20) PRIMARY KEY,
    sale_id INT,
    account_id VARCHAR(20) NOT NULL,
    opportunity_name VARCHAR(200) NOT NULL,
    stage_name VARCHAR(100) NOT NULL,
    amount DECIMAL(15,2) NOT NULL,
    probability DECIMAL(5,2),
    close_date DATE,
    created_date DATE,
    lead_source VARCHAR(100),
    type VARCHAR(100),
    campaign_id INT
);

CREATE OR REPLACE TABLE sf_contacts (
    contact_id VARCHAR(20) PRIMARY KEY,
    opportunity_id VARCHAR(20) NOT NULL,
    account_id VARCHAR(20) NOT NULL,
    first_name VARCHAR(100),
    last_name VARCHAR(100),
    email VARCHAR(200),
    phone VARCHAR(50),
    title VARCHAR(100),
    department VARCHAR(100),
    lead_source VARCHAR(100),
    campaign_no INT,
    created_date DATE
);

---

## 7. Data Loading from Internal Stage

### Loading Data with COPY INTO

Now that the CSV files are in the internal stage (`@ENERGY_STAGE/demo_data/`), we use SQL `COPY INTO` commands to load them into the tables we created earlier.

**Benefits of this approach:**
- Native SQL - simple and performant
- Uses the pre-defined `CSV_FORMAT` file format
- `ON_ERROR = 'CONTINUE'` handles minor data issues gracefully

In [ ]:
# Load Dimension Tables from internal stage
dim_tables = [
    "product_category_dim", "product_dim", "customer_dim", "vendor_dim",
    "account_dim", "department_dim", "region_dim", "sales_rep_dim",
    "campaign_dim", "channel_dim", "employee_dim", "job_dim", "location_dim"
]

print("Loading Dimension Tables...\n")
for table in dim_tables:
    result = session.sql(f"COPY INTO {table} FROM @ENERGY_STAGE/demo_data/{table}.csv FILE_FORMAT = CSV_FORMAT ON_ERROR = 'CONTINUE'").collect()
    rows_loaded = result[0]['rows_loaded'] if result else 0
    print(f"  ✓ {table}: {rows_loaded} rows loaded")

print("\n✓ All dimension tables loaded")

In [ ]:
# Load Fact Tables from internal stage
fact_tables = [
    "sales_fact", "billing_history", "service_logs",
    "finance_transactions", "marketing_campaign_fact", "hr_employee_fact",
    "customer_products"
]

print("Loading Fact Tables...\n")
for table in fact_tables:
    result = session.sql(f"COPY INTO {table} FROM @ENERGY_STAGE/demo_data/{table}.csv FILE_FORMAT = CSV_FORMAT ON_ERROR = 'CONTINUE'").collect()
    rows_loaded = result[0]['rows_loaded'] if result else 0
    print(f"  ✓ {table}: {rows_loaded} rows loaded")

print("\n✓ All fact tables loaded")

In [ ]:
# Load Salesforce CRM Tables from internal stage
sf_tables = ["sf_accounts", "sf_opportunities", "sf_contacts"]

print("Loading Salesforce CRM Tables...\n")
for table in sf_tables:
    result = session.sql(f"COPY INTO {table} FROM @ENERGY_STAGE/demo_data/{table}.csv FILE_FORMAT = CSV_FORMAT ON_ERROR = 'CONTINUE'").collect()
    rows_loaded = result[0]['rows_loaded'] if result else 0
    print(f"  ✓ {table}: {rows_loaded} rows loaded")

print("\n✓ All Salesforce tables loaded")

In [ ]:
%%sql -r verify_load_result
-- Verify data loading
SELECT 'DIMENSION TABLES' as category, '' as table_name, NULL as row_count
UNION ALL SELECT '', 'product_category_dim', COUNT(*) FROM product_category_dim
UNION ALL SELECT '', 'product_dim', COUNT(*) FROM product_dim
UNION ALL SELECT '', 'customer_dim', COUNT(*) FROM customer_dim
UNION ALL SELECT '', 'vendor_dim', COUNT(*) FROM vendor_dim
UNION ALL SELECT '', 'region_dim', COUNT(*) FROM region_dim
UNION ALL SELECT '', '', NULL
UNION ALL SELECT 'FACT TABLES', '', NULL
UNION ALL SELECT '', 'sales_fact', COUNT(*) FROM sales_fact
UNION ALL SELECT '', 'billing_history', COUNT(*) FROM billing_history
UNION ALL SELECT '', 'service_logs', COUNT(*) FROM service_logs
UNION ALL SELECT '', 'finance_transactions', COUNT(*) FROM finance_transactions
UNION ALL SELECT '', 'marketing_campaign_fact', COUNT(*) FROM marketing_campaign_fact
UNION ALL SELECT '', 'hr_employee_fact', COUNT(*) FROM hr_employee_fact;

---

## 8. Semantic Views for Cortex Analyst

### What are Semantic Views?

**Semantic Views** bridge the gap between technical data models and business understanding. They enable:

- **Natural Language Queries** - Ask questions in plain German or English
- **Business Vocabulary** - Define synonyms (e.g., "Kunde" = "Customer")
- **Pre-defined Metrics** - Common calculations ready to use
- **Relationship Mapping** - Automatic joins between tables

### How Cortex Analyst Uses Semantic Views

```
┌─────────────────────────────────────────────────────────────┐
│  User Question: "What were total sales by region last Q?"  │
└─────────────────────────────────────────────────────────────┘
                              │
                              ▼
               ┌──────────────────────────┐
               │     CORTEX ANALYST       │
               │  ┌────────────────────┐  │
               │  │   Semantic View    │  │
               │  │  (Business Logic)  │  │
               │  └────────────────────┘  │
               └──────────────────────────┘
                              │
                              ▼
               ┌──────────────────────────┐
               │    Generated SQL Query   │
               └──────────────────────────┘
```

### Our 4 Semantic Views

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  ENERGY_SALES_SEMANTIC_VIEW                                                 │
│  Analyzes: Contracts, Products, Customers, Regions                          │
├─────────────────────────────────────────────────────────────────────────────┤
│  ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────┐              │
│  │  customer_dim   │  │   product_dim   │  │   region_dim    │              │
│  └────────┬────────┘  └────────┬────────┘  └────────┬────────┘              │
│           │                    │                    │                       │
│           └────────────────────┼────────────────────┘                       │
│                                ▼                                            │
│                    ┌───────────────────────┐                                │
│                    │      sales_fact       │                                │
│                    └───────────────────────┘                                │
│           ┌────────────────────┼────────────────────┐                       │
│           ▼                    ▼                    ▼                       │
│  ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────┐              │
│  │  sales_rep_dim  │  │   vendor_dim    │  │ product_cat_dim │              │
│  └─────────────────┘  └─────────────────┘  └─────────────────┘              │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  BILLING_SEMANTIC_VIEW                                                      │
│  Analyzes: Energy Consumption, Invoices, Payment Status                     │
├─────────────────────────────────────────────────────────────────────────────┤
│           ┌─────────────────┐                                               │
│           │  customer_dim   │                                               │
│           └────────┬────────┘                                               │
│                    │                                                        │
│                    ▼                                                        │
│         ┌─────────────────────┐                                             │
│         │   billing_history   │                                             │
│         └─────────────────────┘                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  SERVICE_SEMANTIC_VIEW                                                      │
│  Analyzes: Service Tickets, Customer Sentiment, Support Channels            │
├─────────────────────────────────────────────────────────────────────────────┤
│           ┌─────────────────┐                                               │
│           │  customer_dim   │                                               │
│           └────────┬────────┘                                               │
│                    │                                                        │
│                    ▼                                                        │
│         ┌─────────────────────┐                                             │
│         │    service_logs     │                                             │
│         └─────────────────────┘                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  HR_SEMANTIC_VIEW                                                           │
│  Analyzes: Employees, Departments, Salaries, Attrition                      │
├─────────────────────────────────────────────────────────────────────────────┤
│  ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────┐              │
│  │ department_dim  │  │  employee_dim   │  │    job_dim      │              │
│  └────────┬────────┘  └────────┬────────┘  └────────┬────────┘              │
│           │                    │                    │                       │
│           └────────────────────┼────────────────────┘                       │
│                                ▼                                            │
│                    ┌───────────────────────┐                                │
│                    │   hr_employee_fact    │                                │
│                    └───────────────────────┘                                │
│                                ▲                                            │
│                    ┌───────────┴───────────┐                                │
│                    │     location_dim      │                                │
│                    └───────────────────────┘                                │
└─────────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
%%sql -r energy_semantic_result
-- ENERGY SALES SEMANTIC VIEW (Contracts and Products)
CREATE OR REPLACE SEMANTIC VIEW ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_SALES_SEMANTIC_VIEW
    tables (
        CUSTOMERS as ENERGY_AI_DEMO.ENERGY_SCHEMA.CUSTOMER_DIM primary key (CUSTOMER_KEY) 
            with synonyms=('Kunden','customers','Privatkunden','Gewerbekunden') 
            comment='German energy customers with housing type for consumption analysis',
        PRODUCTS as ENERGY_AI_DEMO.ENERGY_SCHEMA.PRODUCT_DIM primary key (PRODUCT_KEY) 
            with synonyms=('Produkte','Tarife','products','tariffs') 
            comment='Energy products: Strom, Gas, Solar, Waermepumpe, Smart Home, E-Mobility',
        PRODUCT_CATEGORIES as ENERGY_AI_DEMO.ENERGY_SCHEMA.PRODUCT_CATEGORY_DIM primary key (CATEGORY_KEY)
            with synonyms=('Kategorien','categories')
            comment='Product categories: Electricity, Gas, Solar, Heat Pumps, Smart Home, E-Mobility',
        REGIONS as ENERGY_AI_DEMO.ENERGY_SCHEMA.REGION_DIM primary key (REGION_KEY) 
            with synonyms=('Regionen','regions','Gebiete') 
            comment='German regions: North, South, West, East',
        CONTRACTS as ENERGY_AI_DEMO.ENERGY_SCHEMA.SALES_FACT primary key (SALE_ID) 
            with synonyms=('Vertraege','contracts','sales','Auftraege') 
            comment='Energy contracts and product sales',
        SALES_REPS as ENERGY_AI_DEMO.ENERGY_SCHEMA.SALES_REP_DIM primary key (SALES_REP_KEY) 
            with synonyms=('Berater','Energieberater','consultants') 
            comment='Energy consultants',
        VENDORS as ENERGY_AI_DEMO.ENERGY_SCHEMA.VENDOR_DIM primary key (VENDOR_KEY) 
            with synonyms=('Partner','Installateure','vendors','suppliers') 
            comment='Installation and service partners'
    )
    relationships (
        PRODUCT_TO_CATEGORY as PRODUCTS(CATEGORY_KEY) references PRODUCT_CATEGORIES(CATEGORY_KEY),
        CONTRACTS_TO_CUSTOMERS as CONTRACTS(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY),
        CONTRACTS_TO_PRODUCTS as CONTRACTS(PRODUCT_KEY) references PRODUCTS(PRODUCT_KEY),
        CONTRACTS_TO_REGIONS as CONTRACTS(REGION_KEY) references REGIONS(REGION_KEY),
        CONTRACTS_TO_SALES_REPS as CONTRACTS(SALES_REP_KEY) references SALES_REPS(SALES_REP_KEY),
        CONTRACTS_TO_VENDORS as CONTRACTS(VENDOR_KEY) references VENDORS(VENDOR_KEY),
        CUSTOMERS_TO_REGIONS as CUSTOMERS(REGION_KEY) references REGIONS(REGION_KEY)
    )
    facts (
        CONTRACTS.CONTRACT_AMOUNT as AMOUNT comment='Contract value in EUR',
        CONTRACTS.CONTRACT_UNITS as UNITS comment='Units sold (kWh or count)',
        CONTRACTS.CONTRACT_RECORD as 1 comment='Count of contracts'
    )
    dimensions (
        CUSTOMERS.CUSTOMER_KEY as CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME as CUSTOMER_NAME with synonyms=('Kundenname','Name') comment='Customer name',
        CUSTOMERS.CUSTOMER_TYPE as CUSTOMER_TYPE with synonyms=('Kundenart','Typ') comment='Privatkunde or Gewerbekunde',
        CUSTOMERS.HOUSING_TYPE as HOUSING_TYPE with synonyms=('Wohnform','Haustyp') comment='Housing type for consumption analysis',
        CUSTOMERS.CITY as CITY with synonyms=('Stadt','Ort') comment='Customer city',
        CUSTOMERS.STATE as STATE with synonyms=('Bundesland') comment='German state',
        PRODUCTS.PRODUCT_KEY as PRODUCT_KEY,
        PRODUCTS.PRODUCT_NAME as PRODUCT_NAME with synonyms=('Produktname','Tarif') comment='Product or tariff name',
        PRODUCTS.VERTICAL as VERTICAL with synonyms=('Bereich','Sparte') comment='Product vertical: Electricity, Gas, Solar, etc.',
        PRODUCT_CATEGORIES.CATEGORY_KEY as CATEGORY_KEY,
        PRODUCT_CATEGORIES.CATEGORY_NAME as CATEGORY_NAME with synonyms=('Kategorie') comment='Product category',
        REGIONS.REGION_KEY as REGION_KEY,
        REGIONS.REGION_NAME as REGION_NAME with synonyms=('Region','Gebiet') comment='German region: North, South, West, East',
        CONTRACTS.SALE_ID as SALE_ID with synonyms=('Vertragsnummer','Contract ID') comment='Contract identifier',
        CONTRACTS.DATE as DATE with synonyms=('Vertragsdatum','Datum','Abschlussdatum') comment='Contract date',
        SALES_REPS.SALES_REP_KEY as SALES_REP_KEY,
        SALES_REPS.REP_NAME as REP_NAME with synonyms=('Berater','Vertriebsmitarbeiter') comment='Sales consultant name',
        VENDORS.VENDOR_KEY as VENDOR_KEY,
        VENDORS.VENDOR_NAME as VENDOR_NAME with synonyms=('Partner','Installateur') comment='Installation partner'
    )
    metrics (
        CONTRACTS.TOTAL_REVENUE as SUM(contracts.contract_amount) comment='Total contract revenue in EUR',
        CONTRACTS.TOTAL_CONTRACTS as COUNT(contracts.contract_record) comment='Total number of contracts',
        CONTRACTS.AVERAGE_CONTRACT_VALUE as AVG(contracts.contract_amount) comment='Average contract value',
        CONTRACTS.TOTAL_UNITS as SUM(contracts.contract_units) comment='Total units sold'
    )
    comment='Semantic view for energy sales analysis - contracts, products, customers';

In [ ]:
%%sql -r billing_semantic_result
-- BILLING AND CONSUMPTION SEMANTIC VIEW
CREATE OR REPLACE SEMANTIC VIEW ENERGY_AI_DEMO.ENERGY_SCHEMA.BILLING_SEMANTIC_VIEW
    tables (
        CUSTOMERS as ENERGY_AI_DEMO.ENERGY_SCHEMA.CUSTOMER_DIM primary key (CUSTOMER_KEY)
            with synonyms=('Kunden','customers')
            comment='Energy customers',
        BILLING as ENERGY_AI_DEMO.ENERGY_SCHEMA.BILLING_HISTORY primary key (BILLING_ID)
            with synonyms=('Rechnungen','Abrechnungen','billing','invoices')
            comment='Monthly energy billing and consumption'
    )
    relationships (
        BILLING_TO_CUSTOMERS as BILLING(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY)
    )
    facts (
        BILLING.CONSUMPTION as CONSUMPTION_KWH comment='Energy consumption in kWh',
        BILLING.BILLING_AMOUNT as AMOUNT comment='Invoice amount in EUR',
        BILLING.BILLING_RECORD as 1 comment='Count of billing records'
    )
    dimensions (
        CUSTOMERS.CUSTOMER_KEY as CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME as CUSTOMER_NAME comment='Customer name',
        CUSTOMERS.HOUSING_TYPE as HOUSING_TYPE comment='Housing type',
        CUSTOMERS.CITY as CITY comment='City',
        BILLING.BILLING_ID as BILLING_ID,
        BILLING.BILLING_DATE as BILLING_DATE with synonyms=('Rechnungsdatum','Abrechnungsdatum') comment='Billing date',
        BILLING.BILLING_TYPE as BILLING_TYPE with synonyms=('Energieart','Art') comment='Electricity or Gas',
        BILLING.PAYMENT_STATUS as PAYMENT_STATUS with synonyms=('Zahlungsstatus','Status') comment='Bezahlt, Offen, Ueberfaellig'
    )
    metrics (
        BILLING.TOTAL_CONSUMPTION as SUM(billing.consumption) comment='Total consumption in kWh',
        BILLING.AVERAGE_CONSUMPTION as AVG(billing.consumption) comment='Average consumption in kWh',
        BILLING.TOTAL_BILLING_AMOUNT as SUM(billing.billing_amount) comment='Total billing amount',
        BILLING.TOTAL_INVOICES as COUNT(billing.billing_record) comment='Total number of invoices'
    )
    comment='Semantic view for billing and consumption analysis';

In [ ]:
%%sql -r service_semantic_result
-- SERVICE LOGS SEMANTIC VIEW
CREATE OR REPLACE SEMANTIC VIEW ENERGY_AI_DEMO.ENERGY_SCHEMA.SERVICE_SEMANTIC_VIEW
    tables (
        CUSTOMERS as ENERGY_AI_DEMO.ENERGY_SCHEMA.CUSTOMER_DIM primary key (CUSTOMER_KEY)
            with synonyms=('Kunden','customers')
            comment='Energy customers',
        SERVICE_TICKETS as ENERGY_AI_DEMO.ENERGY_SCHEMA.SERVICE_LOGS primary key (LOG_ID)
            with synonyms=('Tickets','Anfragen','service requests','Kundenservice')
            comment='Customer service tickets and support requests'
    )
    relationships (
        SERVICE_TO_CUSTOMERS as SERVICE_TICKETS(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY)
    )
    facts (
        SERVICE_TICKETS.TICKET_RECORD as 1 comment='Count of service tickets'
    )
    dimensions (
        CUSTOMERS.CUSTOMER_KEY as CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME as CUSTOMER_NAME comment='Customer name',
        CUSTOMERS.CITY as CITY comment='City',
        SERVICE_TICKETS.LOG_ID as LOG_ID,
        SERVICE_TICKETS.LOG_DATE as LOG_DATE with synonyms=('Datum','Erstelldatum') comment='Ticket creation date',
        SERVICE_TICKETS.TOPIC as TOPIC with synonyms=('Thema','Betreff') comment='Topic: Smart Meter, Rechnung, Waermepumpe, Solar, Tarif, Wallbox, Allgemein',
        SERVICE_TICKETS.CATEGORY as CATEGORY with synonyms=('Kategorie') comment='Category: Installation, Abrechnung, Technisch, Vertrag, E-Mobility, Service',
        SERVICE_TICKETS.DESCRIPTION as DESCRIPTION with synonyms=('Beschreibung','Details') comment='Ticket description',
        SERVICE_TICKETS.SENTIMENT as SENTIMENT with synonyms=('Stimmung','Bewertung') comment='Customer sentiment: Positiv, Neutral, Negativ',
        SERVICE_TICKETS.CHANNEL as CHANNEL with synonyms=('Kanal','Kontaktweg') comment='Contact channel: Telefon, Email, Chat, App',
        SERVICE_TICKETS.PRIORITY as PRIORITY with synonyms=('Prioritaet','Dringlichkeit') comment='Priority: Hoch, Mittel, Niedrig',
        SERVICE_TICKETS.RESOLUTION_DATE as RESOLUTION_DATE with synonyms=('Loesungsdatum') comment='Resolution date'
    )
    metrics (
        SERVICE_TICKETS.TOTAL_TICKETS as COUNT(service_tickets.ticket_record) comment='Total service tickets',
        SERVICE_TICKETS.NEGATIVE_TICKETS as SUM(CASE WHEN service_tickets.sentiment = 'Negativ' THEN 1 ELSE 0 END) comment='Negative sentiment tickets'
    )
    comment='Semantic view for customer service analysis';

In [ ]:
%%sql -r customer_energy_semantic_result
-- CUSTOMER ENERGY SEMANTIC VIEW (Cross-Domain: Billing + Products)
-- Enables queries like "average consumption for customers with heat pumps in Hamburg"
CREATE OR REPLACE SEMANTIC VIEW ENERGY_AI_DEMO.ENERGY_SCHEMA.CUSTOMER_ENERGY_SEMANTIC_VIEW
    tables (
        CUSTOMERS as ENERGY_AI_DEMO.ENERGY_SCHEMA.CUSTOMER_DIM primary key (CUSTOMER_KEY)
            with synonyms=('Kunden','customers','Privatkunden','Gewerbekunden')
            comment='German energy customers with housing type and location',
        BILLING as ENERGY_AI_DEMO.ENERGY_SCHEMA.BILLING_HISTORY primary key (BILLING_ID)
            with synonyms=('Rechnungen','Abrechnungen','billing','invoices','Verbrauch')
            comment='Monthly energy billing and consumption data',
        CUSTOMER_PRODUCTS as ENERGY_AI_DEMO.ENERGY_SCHEMA.CUSTOMER_PRODUCTS primary key (CUSTOMER_PRODUCT_ID)
            with synonyms=('Kundenprodukte','owned products','Produktbesitz')
            comment='Products owned by customers - links to product categories',
        PRODUCTS as ENERGY_AI_DEMO.ENERGY_SCHEMA.PRODUCT_DIM primary key (PRODUCT_KEY)
            with synonyms=('Produkte','Tarife','products')
            comment='Energy products: Strom, Gas, Solar, Waermepumpe, Smart Home, E-Mobility',
        PRODUCT_CATEGORIES as ENERGY_AI_DEMO.ENERGY_SCHEMA.PRODUCT_CATEGORY_DIM primary key (CATEGORY_KEY)
            with synonyms=('Kategorien','categories')
            comment='Product categories: Electricity, Gas, Solar, Heat Pumps, Smart Home, E-Mobility'
    )
    relationships (
        BILLING_TO_CUSTOMERS as BILLING(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY),
        CUSTOMER_PRODUCTS_TO_CUSTOMERS as CUSTOMER_PRODUCTS(CUSTOMER_KEY) references CUSTOMERS(CUSTOMER_KEY),
        CUSTOMER_PRODUCTS_TO_PRODUCTS as CUSTOMER_PRODUCTS(PRODUCT_KEY) references PRODUCTS(PRODUCT_KEY),
        PRODUCTS_TO_CATEGORIES as PRODUCTS(CATEGORY_KEY) references PRODUCT_CATEGORIES(CATEGORY_KEY)
    )
    facts (
        BILLING.CONSUMPTION as CONSUMPTION_KWH comment='Energy consumption in kWh',
        BILLING.BILLING_AMOUNT as AMOUNT comment='Invoice amount in EUR',
        BILLING.BILLING_RECORD as 1 comment='Count of billing records',
        CUSTOMER_PRODUCTS.PRODUCT_OWNERSHIP as 1 comment='Product ownership record count'
    )
    dimensions (
        CUSTOMERS.CUSTOMER_KEY as CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME as CUSTOMER_NAME comment='Customer name',
        CUSTOMERS.CUSTOMER_TYPE as CUSTOMER_TYPE with synonyms=('Kundentyp') comment='Privatkunde, Kleingewerbe, Gewerbekunde',
        CUSTOMERS.HOUSING_TYPE as HOUSING_TYPE with synonyms=('Wohnform','Gebaeudetyp') comment='Einfamilienhaus, Reihenhaus, Wohnung, etc.',
        CUSTOMERS.CITY as CITY with synonyms=('Stadt','Ort') comment='City name',
        CUSTOMERS.STATE as STATE with synonyms=('Region','Bundesland') comment='North, South, West, East',
        BILLING.BILLING_ID as BILLING_ID,
        BILLING.BILLING_DATE as BILLING_DATE with synonyms=('Rechnungsdatum','Abrechnungsdatum') comment='Billing date',
        BILLING.BILLING_TYPE as BILLING_TYPE with synonyms=('Energieart','Art') comment='Electricity or Gas',
        BILLING.PAYMENT_STATUS as PAYMENT_STATUS with synonyms=('Zahlungsstatus','Status') comment='Bezahlt, Offen, Ueberfaellig',
        CUSTOMER_PRODUCTS.CUSTOMER_PRODUCT_ID as CUSTOMER_PRODUCT_ID,
        CUSTOMER_PRODUCTS.CATEGORY_NAME as OWNED_CATEGORY with synonyms=('Produktkategorie','besitzt') comment='Category of product owned: Heat Pumps, Solar, E-Mobility, etc.',
        CUSTOMER_PRODUCTS.STATUS as PRODUCT_STATUS comment='Active or Inactive',
        PRODUCTS.PRODUCT_KEY as PRODUCT_KEY,
        PRODUCTS.PRODUCT_NAME as PRODUCT_NAME with synonyms=('Produktname','Tarifname') comment='Product name',
        PRODUCT_CATEGORIES.CATEGORY_KEY as CATEGORY_KEY,
        PRODUCT_CATEGORIES.CATEGORY_NAME as CATEGORY_NAME comment='Product category name'
    )
    metrics (
        BILLING.TOTAL_CONSUMPTION as SUM(billing.consumption) comment='Total consumption in kWh',
        BILLING.AVERAGE_CONSUMPTION as AVG(billing.consumption) comment='Average consumption in kWh',
        BILLING.TOTAL_BILLING_AMOUNT as SUM(billing.billing_amount) comment='Total billing amount in EUR',
        BILLING.TOTAL_INVOICES as COUNT(billing.billing_record) comment='Total number of invoices',
        CUSTOMER_PRODUCTS.TOTAL_PRODUCT_OWNERS as COUNT(DISTINCT customer_products.customer_key) comment='Number of customers owning products'
    )
    comment='Cross-domain semantic view combining billing consumption with product ownership. Use for queries like average consumption for heat pump customers.';

In [ ]:
%%sql -r hr_semantic_result
-- HR SEMANTIC VIEW
CREATE OR REPLACE SEMANTIC VIEW ENERGY_AI_DEMO.ENERGY_SCHEMA.HR_SEMANTIC_VIEW
    tables (
        DEPARTMENTS as ENERGY_AI_DEMO.ENERGY_SCHEMA.DEPARTMENT_DIM primary key (DEPARTMENT_KEY)
            with synonyms=('Abteilungen','departments')
            comment='Company departments',
        EMPLOYEES as ENERGY_AI_DEMO.ENERGY_SCHEMA.EMPLOYEE_DIM primary key (EMPLOYEE_KEY)
            with synonyms=('Mitarbeiter','employees','Personal')
            comment='Employees',
        HR_RECORDS as ENERGY_AI_DEMO.ENERGY_SCHEMA.HR_EMPLOYEE_FACT primary key (HR_FACT_ID)
            with synonyms=('HR-Daten','hr records')
            comment='HR fact records',
        JOBS as ENERGY_AI_DEMO.ENERGY_SCHEMA.JOB_DIM primary key (JOB_KEY)
            with synonyms=('Stellen','jobs','Positionen')
            comment='Job positions',
        LOCATIONS as ENERGY_AI_DEMO.ENERGY_SCHEMA.LOCATION_DIM primary key (LOCATION_KEY)
            with synonyms=('Standorte','locations')
            comment='Office locations'
    )
    relationships (
        HR_TO_DEPARTMENTS as HR_RECORDS(DEPARTMENT_KEY) references DEPARTMENTS(DEPARTMENT_KEY),
        HR_TO_EMPLOYEES as HR_RECORDS(EMPLOYEE_KEY) references EMPLOYEES(EMPLOYEE_KEY),
        HR_TO_JOBS as HR_RECORDS(JOB_KEY) references JOBS(JOB_KEY),
        HR_TO_LOCATIONS as HR_RECORDS(LOCATION_KEY) references LOCATIONS(LOCATION_KEY)
    )
    facts (
        HR_RECORDS.ATTRITION as ATTRITION_FLAG comment='1 = employee left, 0 = active',
        HR_RECORDS.EMPLOYEE_SALARY as SALARY comment='Salary in EUR',
        HR_RECORDS.HR_RECORD as 1 comment='HR record count'
    )
    dimensions (
        DEPARTMENTS.DEPARTMENT_KEY as DEPARTMENT_KEY,
        DEPARTMENTS.DEPARTMENT_NAME as DEPARTMENT_NAME with synonyms=('Abteilung') comment='Department name',
        EMPLOYEES.EMPLOYEE_KEY as EMPLOYEE_KEY,
        EMPLOYEES.EMPLOYEE_NAME as EMPLOYEE_NAME with synonyms=('Mitarbeitername') comment='Employee name',
        EMPLOYEES.GENDER as GENDER with synonyms=('Geschlecht') comment='M or F',
        EMPLOYEES.HIRE_DATE as HIRE_DATE with synonyms=('Eintrittsdatum') comment='Hire date',
        HR_RECORDS.HR_FACT_ID as HR_FACT_ID,
        HR_RECORDS.DATE as DATE with synonyms=('Datum') comment='Record date',
        JOBS.JOB_KEY as JOB_KEY,
        JOBS.JOB_TITLE as JOB_TITLE with synonyms=('Position','Stelle') comment='Job title',
        JOBS.JOB_LEVEL as JOB_LEVEL with synonyms=('Level','Stufe') comment='Job level',
        LOCATIONS.LOCATION_KEY as LOCATION_KEY,
        LOCATIONS.LOCATION_NAME as LOCATION_NAME with synonyms=('Standort') comment='Office location'
    )
    metrics (
        HR_RECORDS.TOTAL_EMPLOYEES as COUNT(hr_records.hr_record) comment='Total employees',
        HR_RECORDS.TOTAL_ATTRITION as SUM(hr_records.attrition) comment='Total attrition',
        HR_RECORDS.AVERAGE_SALARY as AVG(hr_records.employee_salary) comment='Average salary',
        HR_RECORDS.TOTAL_SALARY as SUM(hr_records.employee_salary) comment='Total salary cost'
    )
    comment='Semantic view for HR analytics';

In [ ]:
%%sql -r show_semantic_result
-- Verify semantic views\n
SHOW SEMANTIC VIEWS;

---

## 9. Unstructured Data Processing

### Concept: Cortex Parse Document

**Cortex Parse** is a Snowflake feature that extracts text and structure from unstructured documents:

- **PDF documents**: Contracts, guides, policies
- **Markdown files**: Documentation, FAQs
- **Layout preservation**: Maintains document structure

**Modes:**
- `OCR`: Optical character recognition for scanned documents
- `LAYOUT`: Preserves document layout and structure (recommended)

This parsed content becomes searchable via **Cortex Search**.

In [ ]:
%%sql -r parse_docs_result
-- Parse PDF and Markdown documents
CREATE OR REPLACE TABLE parsed_content AS 
SELECT 
    relative_path,
    BUILD_STAGE_FILE_URL('@ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_STAGE', relative_path) as file_url,
    TO_FILE(BUILD_STAGE_FILE_URL('@ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_STAGE', relative_path)) file_object,
    SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
        @ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_STAGE,
        relative_path,
        {'mode':'LAYOUT'}
    ):content::string as content
FROM directory(@ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_STAGE) 
WHERE relative_path ILIKE 'unstructured_docs/%.pdf' 
   OR relative_path ILIKE 'unstructured_docs/%.md';

In [ ]:
%%sql -r verify_parsed_result
-- Verify parsed documents
SELECT relative_path, LEFT(content, 200) as content_preview 
FROM parsed_content 
LIMIT 100;

---

## 10. Cortex Search Services

### What is Cortex Search?

**Cortex Search** is Snowflake's fully managed, low-latency search engine that powers intelligent search experiences over your data. It's the foundation for building **Retrieval Augmented Generation (RAG)** applications and enterprise search solutions.

### Key Capabilities

- **Hybrid Search** - Combines vector (semantic) and keyword (lexical) search for superior results
- **Automatic Embeddings** - Converts text to vectors using Snowflake Arctic embedding models
- **Semantic Reranking** - AI-powered reranking for optimal relevance
- **Auto-Refresh** - Automatically syncs with source data changes
- **Zero Infrastructure** - No embedding pipelines, vector databases, or tuning required

### How It Works

```
┌────────────────────────────────────────────────────────────────────────┐
│                         CORTEX SEARCH                                  │
├────────────────────────────────────────────────────────────────────────┤
│                                                                        │
│   User Query: "Wie installiere ich eine Wärmepumpe?"                   │
│                              │                                         │
│                              ▼                                         │
│   ┌──────────────────────────────────────────────────┐                │
│   │           HYBRID RETRIEVAL ENGINE                 │                │
│   │  ┌─────────────────┐  ┌─────────────────┐        │                │
│   │  │  Vector Search  │  │ Keyword Search  │        │                │
│   │  │   (Semantic)    │  │   (Lexical)     │        │                │
│   │  └────────┬────────┘  └────────┬────────┘        │                │
│   │           └──────────┬─────────┘                 │                │
│   │                      ▼                           │                │
│   │            ┌─────────────────┐                   │                │
│   │            │ Semantic Rerank │                   │                │
│   │            └────────┬────────┘                   │                │
│   └──────────────────────────────────────────────────┘                │
│                              │                                         │
│                              ▼                                         │
│   Ranked Results: [Heat_Pump_Efficiency_Guide.pdf, ...]               │
│                                                                        │
└────────────────────────────────────────────────────────────────────────┘
```

### Our Cortex Search Services

We create **4 search services** to power our RAG-enabled AI agent:

| Search Service | Content | Use Case |
|----------------|---------|----------|
| `ENERGY_DOCS_SEARCH` | Energy policies, green power terms | Policy & compliance questions |
| `PRODUCT_DOCS_SEARCH` | Product guides, installation manuals | Technical support |
| `SERVICE_DOCS_SEARCH` | Customer service handbook, FAQs | Support agent assistance |
| `SERVICE_LOGS_SEARCH` | Historical service tickets | Similar issue lookup |

### Key Parameters

| Parameter | Description |
|-----------|-------------|
| `ON` | The text column to search |
| `ATTRIBUTES` | Metadata columns to return with results |
| `TARGET_LAG` | How often to refresh the index (e.g., '1 hour') |
| `EMBEDDING_MODEL` | Model for vectorization (default: snowflake-arctic-embed-m-v1.5) |

In [ ]:
%%sql -r search_energy_result
-- Search service for energy policy documents
CREATE OR REPLACE CORTEX SEARCH SERVICE Search_energy_docs
    ON content
    ATTRIBUTES relative_path, file_url, title
    WAREHOUSE = ENERGY_INTELLIGENCE_DEMO_WH
    TARGET_LAG = '30 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            relative_path,
            file_url,
            REGEXP_SUBSTR(relative_path, '[^/]+$') as title,
            content
        FROM parsed_content
        WHERE relative_path ILIKE '%/energy/%'
    );

In [ ]:
%%sql -r search_product_result
-- Search service for product documents
CREATE OR REPLACE CORTEX SEARCH SERVICE Search_product_docs
    ON content
    ATTRIBUTES relative_path, file_url, title
    WAREHOUSE = ENERGY_INTELLIGENCE_DEMO_WH
    TARGET_LAG = '30 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            relative_path,
            file_url,
            REGEXP_SUBSTR(relative_path, '[^/]+$') as title,
            content
        FROM parsed_content
        WHERE relative_path ILIKE '%/products/%'
    );

In [ ]:
%%sql -r search_service_result
-- Search service for customer service documents
CREATE OR REPLACE CORTEX SEARCH SERVICE Search_service_docs
    ON content
    ATTRIBUTES relative_path, file_url, title
    WAREHOUSE = ENERGY_INTELLIGENCE_DEMO_WH
    TARGET_LAG = '30 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            relative_path,
            file_url,
            REGEXP_SUBSTR(relative_path, '[^/]+$') as title,
            content
        FROM parsed_content
        WHERE relative_path ILIKE '%/service/%'
    );

In [ ]:
%%sql -r search_logs_result
-- Search service for service logs (structured data search)
CREATE OR REPLACE CORTEX SEARCH SERVICE Search_service_logs
    ON description
    ATTRIBUTES log_id, customer_key, topic, category, sentiment, priority
    WAREHOUSE = ENERGY_INTELLIGENCE_DEMO_WH
    TARGET_LAG = '1 day'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT
            log_id,
            customer_key,
            topic,
            category,
            description,
            sentiment,
            priority
        FROM service_logs
    );

In [ ]:
%%sql -r show_search_result
-- Verify search services\nSHOW CORTEX SEARCH SERVICES;

---

## 11. Snowflake Intelligence Agent

### What is a Cortex Agent?

**Cortex Agents** are AI orchestrators that plan tasks, use tools to execute them, and generate natural language responses. They combine the power of Large Language Models (LLMs) with your enterprise data to deliver intelligent, context-aware insights.

### Key Capabilities

- **Multi-Tool Orchestration** - Seamlessly switches between structured data (Cortex Analyst) and unstructured data (Cortex Search)
- **Intelligent Planning** - Breaks complex questions into subtasks and routes to the right tools
- **Reflection & Refinement** - Evaluates results after each tool use to improve accuracy
- **Custom Tools** - Extend with stored procedures and UDFs for custom business logic
- **Governed Access** - Respects Snowflake's RBAC, row-access policies, and column-level security

### How Cortex Agents Work

```
┌────────────────────────────────────────────────────────────────────────────┐
│                           CORTEX AGENT WORKFLOW                            │
├────────────────────────────────────────────────────────────────────────────┤
│                                                                            │
│  1. PLANNING                                                               │
│     User Question ──► LLM Orchestrator ──► Task Decomposition              │
│                                                                            │
│  2. TOOL SELECTION & EXECUTION                                             │
│     ┌──────────────────┐  ┌──────────────────┐  ┌──────────────────┐       │
│     │  CORTEX ANALYST  │  │  CORTEX SEARCH   │  │   CUSTOM TOOLS   │       │
│     │   (Text-to-SQL)  │  │      (RAG)       │  │ (Functions/SPs)  │       │
│     │                  │  │                  │  │                  │       │
│     │  Semantic Views  │  │  Document Search │  │  e.g. Send Email │       │
│     │  Structured Data │  │ Unstructured Data│  │  Business Logic  │       │
│     └────────┬─────────┘  └────────┬─────────┘  └────────┬─────────┘       │
│              │                     │                     │                 │
│  3. REFLECTION                     │                     │                 │
│     ◄────────┴─────────────────────┴─────────────────────┘                 │
│     Evaluate results ──► Iterate if needed ──► Generate response           │
│                                                                            │
│  4. RESPONSE                                                               │
│     Natural language answer with citations, tables, or visualizations      │
│                                                                            │
└────────────────────────────────────────────────────────────────────────────┘
```

### Agent Components

| Component | Description |
|-----------|-------------|
| **Instructions** | Define agent personality, response language, and behavior guidelines |
| **Tools** | Cortex Analyst (semantic views), Cortex Search (documents), Custom (UDFs/SPs) |
| **Tool Resources** | Configuration linking tools to specific data sources |
| **Threads** | Persist conversation context across multiple interactions |
| **Sample Questions** | Seed questions to help users get started |

In [ ]:
%%sql -r network_rule_result
/*
-- COMMENTED OUT: External Access for Web Scraping (not needed for simplified demo)
-- Uncomment if you want to enable web scraping capabilities

-- Create network rule for web access
CREATE OR REPLACE NETWORK RULE Energy_Intelligence_WebAccessRule
    MODE = EGRESS
    TYPE = HOST_PORT
    VALUE_LIST = ('0.0.0.0:80', '0.0.0.0:443');
*/

SELECT 'External access setup skipped (web scraping not enabled)' as status;

In [ ]:
%%sql -r ext_access_result
/*
-- COMMENTED OUT: External Access Integration (not needed for simplified demo)
-- Uncomment if you want to enable web scraping capabilities

USE ROLE accountadmin;

GRANT ALL PRIVILEGES ON DATABASE ENERGY_AI_DEMO TO ROLE ACCOUNTADMIN;
GRANT ALL PRIVILEGES ON SCHEMA ENERGY_AI_DEMO.ENERGY_SCHEMA TO ROLE ACCOUNTADMIN;
GRANT USAGE ON NETWORK RULE ENERGY_AI_DEMO.ENERGY_SCHEMA.Energy_Intelligence_WebAccessRule TO ROLE accountadmin;

USE SCHEMA ENERGY_AI_DEMO.ENERGY_SCHEMA;

CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION Energy_Intelligence_ExternalAccess_Integration
    ALLOWED_NETWORK_RULES = (Energy_Intelligence_WebAccessRule)
    ENABLED = true;

GRANT USAGE ON INTEGRATION Energy_Intelligence_ExternalAccess_Integration TO ROLE Energy_Intelligence_Demo;
*/

SELECT 'External access integration skipped (web scraping not enabled)' as status;

In [ ]:
%%sql -r slack_secret_result
/*
-- COMMENTED OUT: Presigned URL Procedure (not needed for simplified demo)
-- Uncomment if you want to enable dynamic document URL generation

USE ROLE Energy_Intelligence_Demo;

CREATE OR REPLACE PROCEDURE Get_File_Presigned_URL_SP(
    RELATIVE_FILE_PATH STRING, 
    EXPIRATION_MINS INTEGER DEFAULT 60
)
RETURNS STRING
LANGUAGE SQL
COMMENT = 'Generates a presigned URL for files in ENERGY_STAGE'
EXECUTE AS CALLER
AS
$$
DECLARE
    presigned_url STRING;
    sql_stmt STRING;
    expiration_seconds INTEGER;
    stage_name STRING DEFAULT '@ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_STAGE';
BEGIN
    expiration_seconds := EXPIRATION_MINS * 60;
    sql_stmt := 'SELECT GET_PRESIGNED_URL(' || stage_name || ', ' || '''' || RELATIVE_FILE_PATH || '''' || ', ' || expiration_seconds || ') AS url';
    EXECUTE IMMEDIATE :sql_stmt;
    SELECT "URL" INTO :presigned_url FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
    RETURN :presigned_url;
END;
$$;
*/

SELECT 'Presigned URL procedure skipped (not needed for simplified demo)' as status;

In [ ]:
%%sql -r web_scrape_result
/*
-- COMMENTED OUT: Web Scraping Function (not needed for simplified demo)
-- Uncomment if you want to enable web scraping capabilities

CREATE OR REPLACE FUNCTION Web_scrape(weburl STRING)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = 3.11
HANDLER = 'get_page'
EXTERNAL_ACCESS_INTEGRATIONS = (Energy_Intelligence_ExternalAccess_Integration)
PACKAGES = ('requests', 'beautifulsoup4')
AS
$$
import requests
from bs4 import BeautifulSoup

def get_page(weburl):
    response = requests.get(weburl)
    soup = BeautifulSoup(response.text)
    return soup.get_text()
$$;
*/

SELECT 'Web scraping function skipped (not needed for simplified demo)' as status;

### Custom Tool: Email Notifications

Enable the agent to send email notifications. This uses Snowflake's built-in `SYSTEM$SEND_EMAIL` functionality.

**Requirements:**
- Email notification integration must exist
- Recipient email must be verified in Snowflake

In [ ]:
%%sql -r email_integration_result
-- Create email notification integration (requires ACCOUNTADMIN)
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE NOTIFICATION INTEGRATION EPOWER_EMAIL_INTEGRATION
    TYPE = EMAIL
    ENABLED = TRUE
    COMMENT = 'Email integration for EPOWER Intelligence Agent';

GRANT USAGE ON INTEGRATION EPOWER_EMAIL_INTEGRATION TO ROLE Energy_Intelligence_Demo;

USE ROLE Energy_Intelligence_Demo;

In [ ]:
%%sql -r email_sp_result
-- Create stored procedure to send emails (Python)
CREATE OR REPLACE PROCEDURE ENERGY_AI_DEMO.ENERGY_SCHEMA.SEND_EMAIL_SP(
    RECIPIENT STRING,
    SUBJECT STRING,
    BODY STRING
)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'send_email'
EXECUTE AS CALLER
AS
$$
from snowflake.snowpark import Session

def send_email(session: Session, recipient: str, subject: str, body: str) -> str:
    session.call(
        'SYSTEM$SEND_EMAIL',
        'EPOWER_EMAIL_INTEGRATION',
        recipient,
        subject,
        body,
        'text/html'
    )
    return f'Email sent to {recipient} with subject: "{subject}"'
$$;

In [ ]:
%%sql -r email_grant_result
-- Grant execute on the email procedure
GRANT USAGE ON PROCEDURE ENERGY_AI_DEMO.ENERGY_SCHEMA.SEND_EMAIL_SP(STRING, STRING, STRING) TO ROLE Energy_Intelligence_Demo;

### Creating the Intelligence Agent

The agent specification defines:
- **Instructions**: How the agent should behave and respond
- **Tools**: Available capabilities (Cortex Analyst, Cortex Search)
- **Tool Resources**: Configuration for each tool

In [ ]:
%%sql -r create_agent_result
-- Create the Snowflake Intelligence Agent
CREATE OR REPLACE AGENT ENERGY_AI_DEMO.ENERGY_SCHEMA.Energy_Chatbot_Agent
WITH PROFILE='{ "display_name": "EPOWER Energy Intelligence Agent" }'
COMMENT=$$ Agent for energy retail analysis: contracts, consumption, service tickets, and documents. $$
FROM SPECIFICATION $$
models:
  orchestration: auto

instructions:
  response: "You are a data analyst for EPOWER Energie Deutschland. CRITICAL LANGUAGE RULE: Always respond in the SAME LANGUAGE as the user's question. If the question is in German, respond entirely in German. If the question is in English, respond entirely in English. Never mix languages in your response. You have access to energy sales data (Strom, Gas, Solar, Heat Pumps, Smart Home, E-Mobility), billing/consumption data, customer service tickets, and internal documents. Provide visualizations when helpful."
  orchestration: "IMPORTANT: Match the language of your response to the user's question language. Use Cortex Search for documents and Cortex Analyst for structured data analysis. For consumption queries combined with products (e.g., heat pump customers, solar customers), use customer_energy_analyst. For simple consumption questions, use billing_analyst. For service tickets, use service_analyst. For product and contract questions, use energy_sales_analyst. If the user wants to send an email, ask for the recipient email address and use send_email."
  sample_questions:
    - question: "Was ist der durchschnittliche Stromverbrauch fuer Kunden mit Waermepumpen in Hamburg?"
    - question: "Zeige mir alle negativen Service-Tickets zum Thema Smart Meter."
    - question: "Welche Produkte wurden letzten Monat am meisten verkauft?"
    - question: "Vergleiche den Verbrauch zwischen Kunden mit und ohne Waermepumpe."

tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: energy_sales_analyst
      description: "Analysiert Energievertraege, Produkte (Strom, Gas, Solar, Waermepumpen), Kunden und Regionen."
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: billing_analyst
      description: "Analysiert Energieverbrauch (kWh), Abrechnungen und Zahlungsstatus fuer Strom und Gas."
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: customer_energy_analyst
      description: "Analysiert Verbrauch kombiniert mit Produktbesitz. Nutze fuer Fragen wie: Verbrauch von Waermepumpen-Kunden, Verbrauch nach Produktkategorie, Kunden mit Solar in Hamburg."
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: service_analyst
      description: "Analysiert Kundenservice-Tickets, Beschwerden und Stimmung nach Thema und Kanal."
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: hr_analyst
      description: "Analysiert Mitarbeiterdaten, Gehaelter, Abteilungen und Fluktuation."
  - tool_spec:
      type: cortex_search
      name: energy_docs_search
      description: "Durchsucht Energierichtlinien, Green Power AGBs und Foerderungsdokumente."
  - tool_spec:
      type: cortex_search
      name: product_docs_search
      description: "Durchsucht Produktanleitungen fuer Solar, Waermepumpen, Smart Meter und E-Mobility."
  - tool_spec:
      type: cortex_search
      name: service_docs_search
      description: "Durchsucht Kundenservice-Handbuch, FAQs und Rechnungserklaerungen."
  - tool_spec:
      type: cortex_search
      name: service_logs_search
      description: "Durchsucht historische Service-Tickets nach aehnlichen Problemen."
  - tool_spec:
      type: data_to_chart
      name: data_to_chart
      description: "Generates visualizations from data."
  - tool_spec:
      type: generic
      name: send_email
      description: "Sendet eine E-Mail an einen Empfaenger. Parameter: recipient (E-Mail-Adresse), subject (Betreff), body (Nachrichtentext als HTML)."
      input_schema:
        type: object
        properties:
          recipient:
            type: string
            description: "E-Mail-Adresse des Empfaengers"
          subject:
            type: string
            description: "Betreff der E-Mail"
          body:
            type: string
            description: "Inhalt der E-Mail (HTML)"
        required:
          - recipient
          - subject
          - body

tool_resources:
  energy_sales_analyst:
    semantic_view: "ENERGY_AI_DEMO.ENERGY_SCHEMA.ENERGY_SALES_SEMANTIC_VIEW"
  billing_analyst:
    semantic_view: "ENERGY_AI_DEMO.ENERGY_SCHEMA.BILLING_SEMANTIC_VIEW"
  customer_energy_analyst:
    semantic_view: "ENERGY_AI_DEMO.ENERGY_SCHEMA.CUSTOMER_ENERGY_SEMANTIC_VIEW"
  service_analyst:
    semantic_view: "ENERGY_AI_DEMO.ENERGY_SCHEMA.SERVICE_SEMANTIC_VIEW"
  hr_analyst:
    semantic_view: "ENERGY_AI_DEMO.ENERGY_SCHEMA.HR_SEMANTIC_VIEW"
  energy_docs_search:
    search_service: "ENERGY_AI_DEMO.ENERGY_SCHEMA.SEARCH_ENERGY_DOCS"
    max_results: 5
  product_docs_search:
    search_service: "ENERGY_AI_DEMO.ENERGY_SCHEMA.SEARCH_PRODUCT_DOCS"
    max_results: 5
  service_docs_search:
    search_service: "ENERGY_AI_DEMO.ENERGY_SCHEMA.SEARCH_SERVICE_DOCS"
    max_results: 5
  service_logs_search:
    search_service: "ENERGY_AI_DEMO.ENERGY_SCHEMA.SEARCH_SERVICE_LOGS"
    max_results: 5
  send_email:
    type: procedure
    identifier: "ENERGY_AI_DEMO.ENERGY_SCHEMA.SEND_EMAIL_SP"
    execution_environment:
      type: warehouse
      warehouse: "ENERGY_INTELLIGENCE_DEMO_WH"
$$;

In [ ]:
%%sql -r grant_role_result
-- Assign agent to Snowflake Intelligence and grant access
USE ROLE accountadmin;

ALTER SNOWFLAKE INTELLIGENCE snowflake_intelligence_object_default ADD AGENT ENERGY_AI_DEMO.ENERGY_SCHEMA.Energy_Chatbot_Agent;
GRANT USAGE ON AGENT ENERGY_AI_DEMO.ENERGY_SCHEMA.Energy_Chatbot_Agent TO ROLE PUBLIC;
GRANT USAGE ON AGENT ENERGY_AI_DEMO.ENERGY_SCHEMA.Energy_Chatbot_Agent TO ROLE Energy_Intelligence_Demo;

USE ROLE Energy_Intelligence_Demo;

---

## 12. Verification & Next Steps

### Setup Complete!

Let's verify everything was created successfully.

In [ ]:
%%sql -r create_agent_result
-- Final verification
SELECT 'Energy AI Demo Setup Complete!' as status;
SELECT 'Database: ENERGY_AI_DEMO.ENERGY_SCHEMA' as info;
SELECT 'Agent: Energy_Chatbot_Agent' as info;
SELECT 'Semantic Views: 4 (Energy Sales, Billing, Service, HR)' as info;
SELECT 'Cortex Search: 4 services (Energy, Product, Service docs + Service logs)' as info;

In [ ]:
%%sql -r grant_agent_result
-- View all created objects
SHOW TABLES IN SCHEMA ENERGY_AI_DEMO.ENERGY_SCHEMA;
SHOW SEMANTIC VIEWS IN SCHEMA ENERGY_AI_DEMO.ENERGY_SCHEMA;
SHOW CORTEX SEARCH SERVICES IN SCHEMA ENERGY_AI_DEMO.ENERGY_SCHEMA;
SHOW AGENTS IN SCHEMA ENERGY_AI_DEMO.ENERGY_SCHEMA;

In [ ]:
%%sql -r test_agent_result
### Try the Agent!

Now you can interact with the Energy Intelligence Agent. Here are some sample questions to try:

**Sales & Products (German):**
- "Welche Produkte wurden letzten Monat am meisten verkauft?"
- "Zeige mir den Umsatz nach Region fuer 2024"

**Consumption & Billing:**
- "What is the average electricity consumption for customers with heat pumps?"
- "Wie ist der Zahlungsstatus unserer Rechnungen aufgeteilt?"

**Service Tickets:**
- "Zeige mir alle negativen Service-Tickets zum Thema Smart Meter"
- "What are the most common service ticket topics?"

**Document Search (RAG):**
- "Was sind die Voraussetzungen fuer die Waermepumpen-Foerderung?"
- "Erklaere mir, wie ich meine Stromrechnung lesen kann"

### What You've Learned

| Concept | What You Did |
|---------|-------------|
| **Git Integration** | Connected to GitHub, loaded data directly from Git stage |
| **Star Schema** | Created dimension and fact tables for analytical workloads |
| **Semantic Views** | Defined business vocabulary, relationships, and metrics for AI |
| **Cortex Parse** | Extracted text from PDF documents |
| **Cortex Search** | Created vector search indexes for RAG |
| **Intelligence Agent** | Built a multi-tool AI assistant |

### Clean Up (Optional)

To remove all demo resources:

```sql
USE ROLE accountadmin;
DROP DATABASE IF EXISTS ENERGY_AI_DEMO;
DROP WAREHOUSE IF EXISTS ENERGY_INTELLIGENCE_DEMO_WH;
DROP ROLE IF EXISTS Energy_Intelligence_Demo;
DROP API INTEGRATION IF EXISTS git_api_integration_energy;
DROP EXTERNAL ACCESS INTEGRATION IF EXISTS Energy_Intelligence_ExternalAccess_Integration;
```